# Étapes 2c et 2d : dictionnaire des données

Ce notebook porte deux étapes de la démarche, qui travaillent sur le même fichier, `data/config/dictionnaire.yaml` :

- étape 2c : le dictionnaire est proposé à partir des données observées, puis fusionné avec le dictionnaire en place, sans perdre aucune décision de revue;
- étape 2d : le métier revoit les propositions dans un classeur Excel (export, import, application, écriture).

Une règle proposée reste au statut `observé` tant que le métier ne l'a pas revue. Les écarts sont signalés dans `a_arbitrer`, et les colonnes sensibles n'exposent aucune valeur ni aucune forme observée.

Chaque mois : mettre à jour `PERIODE_FIN`, puis exécuter les cellules jusqu'à `# ECRITURE APRES FUSION`. Les quatre dernières cellules ne servent qu'à une revue.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from frigeo import racine_projet
from frigeo.perimetre import decouvrir

# PARAMETRE A MODIFIER CHAQUE MOIS
# Dernier mois déposé dans data/raw, au format AAAA-MM
PERIODE_FIN = "2026-03"

In [ ]:
DATA = racine_projet() / "data"
perimetre = decouvrir(DATA, PERIODE_FIN)
perimetre.verifier()
print(perimetre.resume())

In [ ]:
%%time
from frigeo.chargement import charger_tout
from frigeo.dictionnaire import generer_dictionnaire

tables = charger_tout(perimetre)
dictionnaire = generer_dictionnaire(tables)
print({table: len(colonnes) for table, colonnes in dictionnaire.items()})
print(sum(len(colonnes) for colonnes in dictionnaire.values()), "colonnes")

In [ ]:
from frigeo.dictionnaire import resumer_dictionnaire

resume = resumer_dictionnaire(dictionnaire)
print(resume["liste_fermee"].sum(), "listes fermées proposées sur", len(resume), "colonnes")
display(resume[resume["liste_fermee"]].drop(columns=["motif_liste"]))

In [ ]:
# FUSION

from frigeo.dictionnaire import charger_dictionnaire, chemin_dictionnaire
from frigeo.fusion import fusionner

if chemin_dictionnaire().exists():
    genere = dictionnaire
    dictionnaire, rapport = fusionner(charger_dictionnaire(), genere)
    print(len(rapport), "événements, dont", (rapport["niveau"] == "à regarder").sum(), "à regarder")
    print("identique à la génération :", dictionnaire == genere)
    display(rapport)
else:
    print("Premier dictionnaire : rien à fusionner")

In [ ]:
# COLONNES CANDIDATES

resume_fusion = resumer_dictionnaire(dictionnaire)
candidates = resume_fusion[resume_fusion["nb_observees"] > 0]
print(len(candidates), "colonnes candidates,", candidates["nb_observees"].sum(), "valeurs observées")
display(candidates[["table", "colonne", "nb_observees", "motif_liste"]])

In [ ]:
# FORMATS

avec_formes = resume_fusion[
    (resume_fusion["nb_formes"] > 0) | (resume_fusion["nb_formes_observees"] > 0)
]
proposes = avec_formes[avec_formes["nb_formes"] > 0]
candidats = avec_formes[avec_formes["nb_formes_observees"] > 0]
print(len(proposes), "formats proposés,", proposes["nb_formes"].sum(), "formes principales,",
      proposes["nb_formes_a_arbitrer"].sum(), "formes à arbitrer")
print(len(candidats), "colonnes candidates à un format,",
      candidats["nb_formes_observees"].sum(), "formes observées")

# Une colonne sensible n'expose aucune forme observée.
sensibles_avec_forme = sum(
    regles["format"]["motif"] == "colonne sensible" and bool(regles["format"]["liste"])
    for colonnes in dictionnaire.values()
    for regles in colonnes.values()
)
print("colonnes sensibles portant une forme :", sensibles_avec_forme)
display(avec_formes[["table", "colonne", "nb_formes", "nb_formes_a_arbitrer",
                     "nb_formes_observees", "motif_format"]])

In [ ]:
# COMPARAISON AVEC LE CHAPITRE 4

# À exécuter après la cellule FUSION : les listes déclarées par le métier ne figurent
# que dans le dictionnaire enregistré, pas dans la génération brute.
ATTENDUES = {
    ("interventions", "categorie"): {
        "Intervention préventive", "Maintenance", "Dépannage", "Contrôle", "Installation",
    },
    ("interventions", "statut"): {"Réalisée", "Annulée", "Reportée"},
    ("interventions", "facturable"): {"O", "N"},
    ("pieces_interventions", "unite"): {"unité", "kg"},
    ("factures_entetes", "type_piece"): {"Facture", "Avoir"},
    ("factures_entetes", "mode_reglement"): {"Virement", "Chèque", "Prélèvement", "Carte"},
    ("factures_lignes", "code_article"): {
        "MO-STD", "MO-URG", "DEPL", "CTRL-F", "PREV-F", "INST-F", "PIECE", "FLUIDE",
    },
    ("factures_lignes", "unite"): {"heure", "forfait", "unité", "kg"},
    ("achats_entetes", "type_piece"): {"Facture", "Avoir"},
    ("achats_lignes", "unite"): {"unité", "kg", "litre", "forfait"},
    ("clients", "type_commerce"): 8,  # libellés exacts non donnés : comparaison en nombre
    ("clients", "contrat_maintenance"): {"O", "N"},
    ("clients", "statut"): {"Actif", "Inactif"},
    ("fournisseurs", "categorie"): {
        "Pièces frigorifiques", "Fluides", "Carburant", "Frais généraux",
    },
    ("fournisseurs", "delai_paiement"): {"30 j fin de mois", "45 j fin de mois"},
    ("fournisseurs", "statut"): {"Actif", "Inactif"},
    ("catalogue_pieces", "famille"): {
        "Compresseur", "Ventilation", "Régulation", "Joint", "Détente", "Résistance",
        "Sonde", "Fluide",
    },
    ("catalogue_pieces", "unite"): {"unité", "kg"},
    ("intervenants", "poste"): {"Technicien frigoriste", "Chef d'équipe", "Apprenti"},
    ("intervenants", "type_contrat"): {"CDI", "CDD", "Apprentissage"},
    ("intervenants", "motif_sortie"): {
        "Démission", "Fin de CDD", "Rupture conventionnelle", "Retraite",
    },
    ("intervenants", "secteur"): {"27", "76"},
}

lignes = []
for (table, colonne), attendu in ATTENDUES.items():
    valeurs = dictionnaire[table][colonne]["valeurs"]
    liste = valeurs["liste"]
    ligne = {"table": table, "colonne": colonne}
    if valeurs["statut"] == "documenté":
        # Liste déclarée par le métier : ses valeurs sont celles qu'il a retenues.
        retenues = {v["valeur"] for v in liste if v["statut"] in ("valide", "documenté")}
        a_arbitrer = set()
        accord, prefixe = "documentée", f"{len(retenues)} valeurs retenues"
    elif valeurs["proposition"] != "liste fermée":
        ligne.update(verdict="non proposée", detail=valeurs["motif"])
        lignes.append(ligne)
        continue
    else:
        retenues = {v["valeur"] for v in liste if v["origine"] == "liste proposée"}
        a_arbitrer = {v["valeur"] for v in liste if v["origine"] == "à arbitrer"}
        accord, prefixe = "identique", ""

    detail = [prefixe] if prefixe else []
    if isinstance(attendu, int):
        verdict = accord if len(retenues) == attendu else "écart"
        detail.append(f"{len(retenues)} valeurs pour {attendu} attendues")
    else:
        manquantes, en_trop = attendu - retenues, retenues - attendu
        if not manquantes and not en_trop:
            verdict = accord
        elif manquantes <= a_arbitrer and not en_trop:
            verdict = "valeur rare à arbitrer"
        else:
            verdict = "écart"
        if manquantes:
            detail.append(f"manquantes {sorted(manquantes)}")
        if en_trop:
            detail.append(f"en trop {sorted(en_trop)}")
    ligne.update(verdict=verdict, detail="; ".join(detail))
    lignes.append(ligne)

comparaison = pd.DataFrame(lignes)
display(comparaison["verdict"].value_counts().rename("colonnes").to_frame())
display(comparaison[comparaison["verdict"] != "identique"])

# Listes proposées par la génération pour des colonnes absentes du chapitre 4.
hors_chapitre = pd.DataFrame(
    [
        {
            "table": table,
            "colonne": colonne,
            "statut": regles["valeurs"]["statut"],
            "regle_en_vigueur": regles["valeurs"]["regle"],
        }
        for table, colonnes in dictionnaire.items()
        for colonne, regles in colonnes.items()
        if regles["valeurs"]["proposition"] == "liste fermée"
        and (table, colonne) not in ATTENDUES
    ]
)
display(hors_chapitre)

## Synthèse de la comparaison avec le chapitre 4

Le chapitre 4 du dossier de données donne les valeurs admises de 22 colonnes. Il tient ici le rôle du métier, et ne sert qu'à vérifier que le circuit retrouve les bonnes listes : chez un client, cette référence n'existe pas.

La comparaison porte sur le dictionnaire après fusion, donc avec les décisions de revue :

- 13 listes sont proposées par la génération à l'identique;
- 2 le sont avec une valeur rare à arbitrer (`Avoir` dans `type_piece` des factures et des achats, soit 0,13 % et 0,27 % des lignes);
- 7 ne sont pas proposées par la génération (6 faute d'effectif, car `fournisseurs` et `intervenants` ont moins de 200 lignes; 1 colonne sensible, `motif_sortie`) et ont été déclarées par le métier à la revue. Leurs valeurs retenues sont exactement celles du chapitre 4 : pour les 6 colonnes non sensibles, des valeurs observées validées sans ressaisie; pour `motif_sortie`, des valeurs saisies à la main, sans effectif.

Aucune valeur en trop, aucun libellé divergent. Trois listes sont proposées hors chapitre 4 : `JournalCode` et `JournalLib` (plausibles, pas encore revues), et `fournisseur_principal` (faux positif, clé vers le référentiel des fournisseurs, écartée à la revue).

Sur un premier dictionnaire, avant toute revue, les 7 colonnes déclarées apparaissent « non proposée ».

## Écriture du dictionnaire

Le dictionnaire est écrit dans `data/config/dictionnaire.yaml`, hors du dépôt, car il contient des valeurs observées dans les données. L'écriture est refusée si une décision de revue du fichier en place serait perdue ou modifiée : une régénération passe donc toujours par la fusion. Un fichier qui porte des décisions est copié dans `data/config/sauvegardes/` avant d'être remplacé.

In [ ]:
# ECRITURE APRES FUSION

from frigeo.dictionnaire import (
    charger_dictionnaire,
    charger_meta,
    dossier_sauvegardes,
    ecrire_dictionnaire,
)
from frigeo.confidentialite import charger_sensibilite

# Sans risque pour les décisions de revue : l'écriture est refusée si l'une d'elles
# serait perdue ou modifiée, et le fichier en place est copié avant d'être remplacé.
chemin = ecrire_dictionnaire(dictionnaire, PERIODE_FIN)
relu = charger_dictionnaire(chemin)

lots = [lot for colonnes in relu.values() for lot in colonnes.values()]
regles = [regle for lot in lots for regle in lot.values()]
valeurs = [valeur for lot in lots for valeur in lot["valeurs"]["liste"]]
formes = [forme for lot in lots for forme in lot["format"]["liste"]]
elements = regles + valeurs + formes
print(chemin)
print(len(relu), "tables,", len(lots), "colonnes,", len(regles), "règles,",
      len(valeurs), "valeurs,", len(formes), "formes")
print("statuts :", sorted({element["statut"] for element in elements}))
print("décisions de revue :", sum(element["statut"] != "observé" for element in elements))
print("identique au dictionnaire fusionné :", relu == dictionnaire)
meta = charger_meta()
print("structure :", meta["structure"])
print("revues au journal :", len(meta["revues"]))

# Une colonne sensible ne porte que des valeurs et des formes déclarées par le métier,
# sans effectif.
sensibles = charger_sensibilite()
lots_sensibles = {
    (table, colonne): relu[table][colonne]
    for table, colonnes in sensibles.items()
    for colonne in colonnes
}
exposees = [
    cle
    for cle, lot in lots_sensibles.items()
    if any(
        element["origine"] != "ajoutée" or element["effectif"] is not None
        for element in lot["valeurs"]["liste"] + lot["format"]["liste"]
    )
]
declarees = sum(bool(lot["valeurs"]["liste"]) for lot in lots_sensibles.values())
print(len(lots_sensibles), "colonnes sensibles, dont", len(exposees),
      "exposant des valeurs observées, des formes observées ou un effectif et", declarees,
      "avec une liste déclarée par le métier")
print(round(chemin.stat().st_size / 1024), "Ko")

copies = sorted(dossier_sauvegardes().glob("*.yaml")) if dossier_sauvegardes().exists() else []
print(len(copies), "copies de sauvegarde, dernière :", copies[-1].name if copies else "aucune")

In [ ]:
# EXPORT

from openpyxl import load_workbook

from frigeo.dictionnaire import charger_dictionnaire, charger_meta
from frigeo.revue import exporter_revue

# Le classeur part toujours du dictionnaire enregistré, avec ses décisions.
dictionnaire_courant = charger_dictionnaire()
nb_revues = len(charger_meta()["revues"])
chemin_classeur = exporter_revue(
    dictionnaire_courant, PERIODE_FIN, nb_revues=nb_revues, ecraser=False
)
print(chemin_classeur)
print("revues déjà appliquées :", nb_revues)
classeur = load_workbook(chemin_classeur, read_only=True)
for nom in ("regles", "valeurs", "formes"):
    lignes = classeur[nom].iter_rows(min_row=2, values_only=True)
    print(nom, sum(any(v is not None for v in ligne) for ligne in lignes), "lignes")
classeur.close()

In [ ]:
# IMPORT

from frigeo.dictionnaire import charger_dictionnaire, charger_meta
from frigeo.revue import chemin_revue, importer_revue

dictionnaire_courant = charger_dictionnaire()
nb_revues = len(charger_meta()["revues"])
chemin_classeur = chemin_revue(PERIODE_FIN)
revue = importer_revue(chemin_classeur, dictionnaire_courant, nb_revues=nb_revues)
print(len(revue.regles), "règles,", len(revue.valeurs), "valeurs,", len(revue.formes), "formes")
print("statuts :", sorted(
    set(revue.regles["statut"]) | set(revue.valeurs["statut"]) | set(revue.formes["statut"])
))
print("remplacements dans le classeur :", int(revue.valeurs["remplacement"].notna().sum()))
print("avertissements :", revue.avertissements)

In [ ]:
# APPLICATION

from frigeo.revue import appliquer_revue, reste_a_traiter

dictionnaire_revu, rapport_revue, entree_revue = appliquer_revue(
    dictionnaire_courant, revue, chemin_classeur
)
if entree_revue is None:
    print("Aucun changement : rien à écrire.")
else:
    print(entree_revue)
    print(rapport_revue["evenement"].value_counts().to_string())

# Ce qui restera à revoir une fois cette revue écrite, sans attendre un nouvel export.
reste = reste_a_traiter(dictionnaire_revu)
print("Reste à traiter après cette revue")
print(f"  règles : {reste['regles_a_traiter']} sur {reste['regles']}, dont "
      f"{reste['regles_a_decider']} à décider et {reste['regles_avec_ecarts']} avec des écarts "
      f"(hors {reste['regles_sans_liste']} lignes « valeurs » sans liste proposée et "
      f"{reste['regles_sans_format']} lignes « format » sans format proposé)")
print(f"  valeurs : {reste['valeurs_a_traiter']} sur {reste['valeurs']}, dont "
      f"{reste['valeurs_a_arbitrer']} à arbitrer (hors {reste['valeurs_listes_ecartees']} "
      f"valeurs de listes écartées et {reste['valeurs_non_declarees']} valeurs de listes "
      "non déclarées)")
print(f"  formes : {reste['formes_a_traiter']} sur {reste['formes']}, dont "
      f"{reste['formes_a_arbitrer']} à arbitrer (hors {reste['formes_formats_ecartes']} "
      f"formes de formats écartés et {reste['formes_non_declarees']} formes de formats "
      "non déclarés)")
print("Revue complète." if reste["complet"] else "Revue partielle : un nouvel export sera utile.")
rapport_revue

In [ ]:
# ECRITURE

from frigeo.dictionnaire import (
    charger_dictionnaire,
    charger_meta,
    dossier_sauvegardes,
    ecrire_dictionnaire,
)
from frigeo.revue import reste_a_traiter

if entree_revue is None:
    print("Aucun changement : dictionnaire non réécrit.")
else:
    # Le fichier en place est copié dans data/config/sauvegardes avant d'être remplacé.
    # Seules les décisions datées de cette revue peuvent changer.
    chemin_ecrit = ecrire_dictionnaire(dictionnaire_revu, PERIODE_FIN, revue=entree_revue)
    assert charger_dictionnaire() == dictionnaire_revu
    print(chemin_ecrit)
    copies = sorted(dossier_sauvegardes().glob("*.yaml"))
    print(len(copies), "copies de sauvegarde, dernière :", copies[-1].name)
    # Une seconde exécution de la cellule ne doit pas journaliser la revue deux fois.
    entree_revue = None

# État du dictionnaire enregistré.
reste = reste_a_traiter(charger_dictionnaire())
print("revues au journal :", len(charger_meta()["revues"]))
print(f"reste à traiter : {reste['regles_a_traiter']} règles sur {reste['regles']} "
      f"(hors {reste['regles_sans_liste']} lignes « valeurs » sans liste proposée et "
      f"{reste['regles_sans_format']} lignes « format » sans format proposé), "
      f"{reste['valeurs_a_traiter']} valeurs sur {reste['valeurs']} "
      f"(hors {reste['valeurs_listes_ecartees']} valeurs de listes écartées et "
      f"{reste['valeurs_non_declarees']} valeurs de listes non déclarées), "
      f"{reste['formes_a_traiter']} formes sur {reste['formes']} "
      f"(hors {reste['formes_formats_ecartes']} formes de formats écartés et "
      f"{reste['formes_non_declarees']} formes de formats non déclarés)")
print("Revue complète." if reste["complet"] else "Revue partielle.")